# Create Model from Training Run for Batch
This notebook enables running one-off batch inferenc jobs. The user will provide a training job name and, optionally, a pointer to the checkpoints weights to use for inference. The user will then provide a customer inference file that contains the specific outputs the user desires. The notebook will create a SageMaker Model for the run. 

The second part of the notebook will take in the input and output parameters for running the batch job.

## Model Parameters
Please fill in the model parameters below

In [1]:
# Provide a unique job name for this batch inference run.
# The job name will be used to name the created model and name the batch job


JOB_NAME = "big-training-100-20-20-split---250624-2327--c2-labeled-data"
JOB_NAME = "big-training-80-20-20-split---251105-2349--c2-labeled-data"

# Name of the SageMaker Training job used to train the model
sm_training_job_name = "c1-cln-c2-1177-c2-25-c2-218-c3-14-100-20-20-split---250624-2327"
sm_training_job_name = "c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--1-it0-251105-2349"

# (OPTIONAL) S3 URI to the checkpoint weights 
# Leave blank to use the best weights as computed during testing during the training run
# NOTE!!! If the training job did not "Complete" (e.g., was "Stopped" or "Failed"), you must provide
# a checkpoint to use for the weights.
model_weights_s3_uri = "s3://mz-temp/c1-cln-c2-1177-c2-25-c2-218-c3-14-100-20-20-split---250624-2327/checkpoints/last.ckpt"
model_weights_s3_uri = "s3://mz-temp/c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--1-it0-251105-2349/checkpoints/last.ckpt"

# (OPTIONAL) Inference Container
# Can optionally point to the pytorch inference container to use for the model hosted in ECR.
# If none provided, will use the SageMaker hosted container with same PyTorch and Python versions
# as used during training.
inference_container_uri = ""

import os
# The SageMaker role to use, if non-provided will use the default SageMaker execution role
role = os.getenv('AWS_ROLE')


## Inference File
Create the inference file to be used for this batch job.

Copy the template inference file `src/batch_inference_template.py` and give the file a unique name. 
Ensure that the new file is also placed in the `src` directory.

The bulk of the logic will be updating the `predict_fn()` and `output_fn()` to perform the desired inference and output the desired content.

If parameters need to be passed for additional calculationgs, we recommend setting them as environment variable in the cell below. Load these variables in the `model_fn()` and store them in the `data_param` dict already created by this function. Note, these parameters will be shared by all instances of the model. Per image parameters should be passed in as input.


In [2]:
# The name of the inference file here. Ensure the file is located under the `src` directory in this repo
inference_filename = "batch_inference_debug.py"

# Update to point to the `src` folder in this directory
src_directory = "../src"

# Add any environment variables you want to pass to the model here
# Can also override default values used when making models here as well
additional_environment_variables = {
    # "EXAMPLE_VARIABLE": "VALUE",
}

# Optional Parameters
Below are some optional model parameters you can set if desired.

Please refer to SageMaker documentation on what these values provide.

In [3]:
# Some default model variables
max_payload_size_mb=25
workers_per_model = 1
env_variables = {
    # Update the maximum payload size for TS. Needs to match SageMaker's max payload size
    # Default for TS is 6,553,500 bytes
    # Convert max_payload_size in MB to Bytes
    "TS_DEFAULT_WORKERS_PER_MODEL": str(workers_per_model),
    "TS_MAX_REQUEST_SIZE": str(max_payload_size_mb * 1024 * 1024)
}

pytorch_version = "2.3"
inference_target_instance = "ml.m5.2xlarge" #"ml.m5.2xlarge"

# Provide the uri for the inference image directly here, otherwise will use SageMaker hosted container
# based on pytorch version.
image_uri = ""

# Build Model
Do not modify unless needed.

Builds the SageMaker model with the following steps:
1. Get training job information
2. Get 
3. 

In [4]:
import torch
from tarfile import TarFile, TarInfo
import boto3
import sagemaker
from sagemaker.pytorch import PyTorchModel
from pathlib import Path
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\zelinski1\AppData\Local\sagemaker\sagemaker\config.yaml


In [5]:
# Some helper variables
job_directory = Path(f"model_creation_{JOB_NAME}")

orig_model_tar_file = job_directory / "orig_model.tar.gz"
model_weights_directory = job_directory / "model"
model_source_directory = job_directory / "source"

model_tar_file = job_directory / "model.tar.gz"
source_tar_file = job_directory / "source.tar.gz"

if job_directory.exists():
    raise ValueError("Folder with job name already used. Please use a different job name or clean up this folder to proceed")

In [6]:
s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")

In [7]:
print(job_directory)
print(orig_model_tar_file)
#sm_training_job_name = "s3://sagemaker-us-west-2-159929462505/17k-dataset-cleaned-1--1-it0-241018-1441"
#sm_training_job_name = "17k-dataset-cleaned-1--1-it0-241018-1441"

#orig_model_tar_file =     "s3://sagemaker-us-west-2-159929462505/17k-dataset-cleaned-1--1-it0-241018-1441/output/model.tar.gz"
#orig_model_tar_file = "model.tar.gz"

print(orig_model_tar_file)


model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data
model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\orig_model.tar.gz
model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\orig_model.tar.gz


In [8]:
model_weights_directory.mkdir(parents=True)

In [9]:
print( (model_weights_directory / "code").exists() )

False


In [10]:
training_job = sm_client.describe_training_job(TrainingJobName=sm_training_job_name)


In [11]:
training_job

{'TrainingJobName': 'c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--1-it0-251105-2349',
 'TrainingJobArn': 'arn:aws:sagemaker:us-west-2:159929462505:training-job/c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--1-it0-251105-2349',
 'ModelArtifacts': {'S3ModelArtifacts': 's3://sagemaker-us-west-2-159929462505/c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--1-it0-251105-2349/output/model.tar.gz'},
 'TrainingJobStatus': 'Completed',
 'SecondaryStatus': 'Completed',
 'HyperParameters': {'batch_size': '12',
  'center_crop': '"True"',
  'center_crop_size': '1200',
  'color_jitter': '0.0',
  'epochs': '25',
  'image_mode': '"grayscale"',
  'image_size': '1200',
  'learning_rate': '0.0001',
  'lr_schedular': '"cosine_warmup"',
  'model_name': '"UNet"',
  'num_workers': '16',
  'project_name': '"wandb-test-count-1"',
  'run_name': '"c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--1-it0-251105-2349"',
  'sagemaker_container_log_level': '20',
  'sagemaker_job_name': '"c1-cln-c2-1176-c2-25-c2-218-c3-14-80-20-20--

In [12]:
# Get the model.tar.gz file associated with the training job if provided
training_job = sm_client.describe_training_job(TrainingJobName=sm_training_job_name)
model_s3_uri = training_job["ModelArtifacts"]["S3ModelArtifacts"]
bucket, key = model_s3_uri.replace("s3://", "").split("/", 1)

s3_client.download_file(bucket, key, orig_model_tar_file)
with TarFile.open(orig_model_tar_file, "r:gz") as tf:
    tf.extractall(model_weights_directory)

print( (model_weights_directory / "code").exists() )

if (model_weights_directory / "code").exists():
    (model_weights_directory / "code").rmdir()

False


In [13]:
# Get the model.tar.gz file associated with the training job if provided
training_job = sm_client.describe_training_job(TrainingJobName=sm_training_job_name)
model_s3_uri = training_job.get("ModelArtifacts", {}).get("S3ModelArtifacts")
if model_s3_uri is not None:
    bucket, key = model_s3_uri.replace("s3://", "").split("/", 1)
    s3_client.download_file(bucket, key, orig_model_tar_file)
    with TarFile.open(orig_model_tar_file, "r:gz") as tf:
        tf.extractall(model_weights_directory)
else:
    source_code_uri = training_job["HyperParameters"]["sagemaker_submit_directory"].strip('"')
    bucket, key = source_code_uri.replace("s3://", "").split("/", 1)
    s3_client.download_file(bucket, key, orig_model_tar_file)
    with TarFile.open(orig_model_tar_file, "r:gz") as tf:
        tf.extractall(model_weights_directory / "code")
    
print( (model_weights_directory / "code").exists() )
if (model_weights_directory / "code").exists():
    (model_weights_directory / "code").rmdir()

False


In [14]:
# If checkpoints weight is given, download the weights
if model_weights_s3_uri:
    bucket, key = model_weights_s3_uri.replace("s3://", "").split("/", 1)
    extension = Path(key).suffix
    s3_client.download_file(bucket, key, model_weights_directory / f"best{extension}")

In [15]:

print(os.getcwd())

print( model_source_directory )

print(  model_source_directory.exists() )

print( model_source_directory / inference_filename )

print( os.listdir( model_source_directory ) )

print( os.listdir(src_directory) ) 



C:\Users\zelinski1\Desktop\work\Thrust2\repo\sm-unet-training-pipeline\notebooks
model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\source
True
model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\source\batch_inference_debug.py
['.ipynb_checkpoints', 'batch_inference_debug.py', 'batch_inference_template.py', 'copy_s3_masks.py', 'crop_masks_on_s3.py', 'inference.py', 'inspect_image_and_mask.py', 'llnl_ml', 'om_inference.py', 'requirements.txt', 's3_inference_examine.py', 'split_up_directory.py', 'train.py', 'Untitled.ipynb', 'wandb', '__pycache__']
['.ipynb_checkpoints', 'batch_inference_debug.py', 'batch_inference_template.py', 'copy_s3_masks.py', 'crop_masks_on_s3.py', 'inference.py', 'inspect_image_and_mask.py', 'llnl_ml', 'om_inference.py', 'requirements.txt', 's3_inference_examine.py', 'split_up_directory.py', 'train.py', 'Untitled.ipynb', 'wandb', '__pycache__']


In [16]:
# Copy over source code and inference filename
if not model_source_directory.exists(): 
    model_source_directory.mkdir()
    

    # If running from Jupyter notebook, there might be a write permission issue. Manual copying might be necessary. 
    shutil.copy(src_directory, model_source_directory)
    
if not (model_source_directory / inference_filename).exists():
    raise ValueError(f"Could not find '{inference_filename}' in source directory '{src_directory}'")

In [17]:
print( source_tar_file )
print( model_tar_file )

model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\source.tar.gz
model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\model.tar.gz


In [18]:
# Tar the model and source directories and place into S3
# Create new tarfile
def filter_temp_dirs(x: TarInfo):
    if "ipynb_checkpoints" in x.name or "__pycache__" in x.name:
        return None
    return x

# Create tarfile with just model weights
with TarFile.open(model_tar_file, "w:gz") as model_tar:
    model_tar.add(model_weights_directory, arcname="", filter=filter_temp_dirs)
    print("Model Tar Contents: ")
    print(model_tar.list())


# Create tarfile with just the source code
with TarFile.open(source_tar_file, "w:gz") as model_tar:
    model_tar.add(model_source_directory, arcname="", filter=filter_temp_dirs)
    print("Source Tar Contents: ")
    print(model_tar.list())

Model Tar Contents: 
drwxrwxrwx 0/0          0 2025-11-06 08:44:33 / 
-rw-rw-rw- 0/0  372429944 2025-11-06 08:44:33 best.ckpt 
-rw-rw-rw- 0/0        172 2025-11-05 23:55:50 config.yaml 
None
Source Tar Contents: 
drwxrwxrwx 0/0          0 2025-11-06 08:43:26 / 
-rw-rw-rw- 0/0        796 2024-05-31 23:15:24 Untitled.ipynb 
-rw-rw-rw- 0/0      11133 2025-03-20 21:28:18 batch_inference_debug.py 
-rw-rw-rw- 0/0       5910 2024-08-05 16:44:01 batch_inference_template.py 
-rw-rw-rw- 0/0       1379 2024-12-18 12:53:46 copy_s3_masks.py 
-rw-rw-rw- 0/0       3672 2024-12-17 16:23:39 crop_masks_on_s3.py 
-rw-rw-rw- 0/0       4145 2024-08-05 16:44:01 inference.py 
-rw-rw-rw- 0/0       1020 2024-07-01 10:09:41 inspect_image_and_mask.py 
drwxrwxrwx 0/0          0 2025-11-06 08:43:26 llnl_ml/ 
-rw-rw-rw- 0/0          0 2024-01-22 22:41:05 llnl_ml/__init__.py 
-rw-rw-rw- 0/0      14022 2025-03-19 12:39:30 llnl_ml/data.py 
-rw-rw-rw- 0/0       8506 2025-03-21 16:27:49 llnl_ml/lightning.py 
-rw-rw-rw- 

In [19]:
bucket = sagemaker.Session().default_bucket()
# Upload the model tar file
s3_client.upload_file(str(model_tar_file), bucket, str(model_tar_file))

# Upload the source tar file
s3_client.upload_file(str(source_tar_file), bucket, str(source_tar_file))

# Create the S3 URIs for each
model_tar_uri = f"s3://{bucket}/{model_tar_file}"
source_tar_uri = f"s3://{bucket}/{source_tar_file}"

In [20]:
# Get the execution role arn if not provided above
role = role or sagemaker.get_execution_role()
if not image_uri:
    image_uri = sagemaker.image_uris.retrieve(framework="PyTorch", region='us-west-2', version=pytorch_version, instance_type=inference_target_instance, image_scope='inference')

print( JOB_NAME )
print( image_uri )
print( model_tar_uri )
print( source_tar_uri )
print( inference_filename )
print( env_variables )

# Create the model
model = PyTorchModel(
    name=JOB_NAME,
    image_uri=image_uri,
    model_data=model_tar_uri,
    role=role or sagemaker.get_execution_role(),
    source_dir=source_tar_uri,
    entry_point=inference_filename,
    env=env_variables,
    sagemaker_session=sagemaker.Session(),
    model_server_workers=workers_per_model
)

model.create()

big-training-80-20-20-split---251105-2349--c2-labeled-data
763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.3-cpu-py311
s3://sagemaker-us-west-2-159929462505/model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\model.tar.gz
s3://sagemaker-us-west-2-159929462505/model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\source.tar.gz
batch_inference_debug.py
{'TS_DEFAULT_WORKERS_PER_MODEL': '1', 'TS_MAX_REQUEST_SIZE': '26214400'}


In [21]:
print( source_tar_uri )

s3://sagemaker-us-west-2-159929462505/model_creation_big-training-80-20-20-split---251105-2349--c2-labeled-data\source.tar.gz


# Sample Image Pair Input
The next section goes over some example data to create a batch job to process an image and mask pair within the model.

## Create Input
A transform job can take in a single s3 object to process within the `input_fn`. To allow passing in pairs of items, we will do a small work around of creating JSON objects to point to these image/mask pairs for the model to load.

This block
1. Finds s3 uris to all images and masks
2. Create json files for each pair
3. Upload files to S3
4. Create manifest file from uploaded json files

### Get Image and Mask URIs

In [22]:
#image_s3_uri_prefix = "s3://baselabelinginfrastack-labelingjobbucket62ec7eaa-1g62nohkpw4rc/campaign3-1747783016/labeled/100-samples-each-layer/" # e.g."s3://bucket/path/to/images/"
image_s3_uri_prefix = "s3://baselabelinginfrastack-alllabeledimages/images_camp2/"
#image_s3_uri_prefix = "s3://databricks-159929462505-comp-24comp005/in/polymerdiw/mongodb-export/2025-05-15/files/"
mask_s3_uri_prefix = "s3://baselabelinginfrastack-alllabeledimages/masks_camp1_cln_camp2_1176_camp2_218_camp2_25_camp3_14/"

In [23]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

def list_s3_files(s3_uri: str, region: str = "us-west-2"):
    s3 = boto3.resource("s3")
    # Split s3 uri into bucket and prefix
    bucket, prefix = s3_uri.removeprefix("s3://").split("/", 1)
    # Add ending "/" if not present on prefix
    prefix = prefix if prefix.endswith("/") else prefix + "/"

    # Get filenames for all objects on the prefix
    filenames = []
    for obj in s3.Bucket(bucket).objects.filter(Prefix=prefix):
        filename = obj.key
        if filename.endswith(IMAGE_EXTENSIONS):
            filenames.append(f"{bucket}/{filename}")

    return sorted(filenames)

In [24]:
image_s3_uris = list_s3_files(image_s3_uri_prefix)
mask_s3_uris = list_s3_files(mask_s3_uri_prefix)

In [25]:

print( [os.path.basename(image_s3_uris[i]) for i in range(10)] )
print( [os.path.basename(mask_s3_uris[i]) for i in range(10)] )

import json
use_json = True
if use_json: 
    json_file_list = r"C:\Users\zelinski1\Desktop\work\Thrust2\repo\sm-unet-training-pipeline/c2_1176_c2_25_c2_218_no_split.json"  
    with open(json_file_list, 'r') as file:
        file_data = json.load(file)

    print( len( file_data['image'] ) )

    image_s3_uris_temp = []
    mask_s3_uris_temp = []

    for i in range(len(file_data['image'])):
        for j in range(len(image_s3_uris)):
            if file_data['image'][i][0:24] in image_s3_uris[j]: 
                image_s3_uris_temp.append( image_s3_uris[j] )

        for j in range(len(mask_s3_uris)):
            if file_data['mask'][i][0:24] in mask_s3_uris[j]:
                mask_s3_uris_temp.append( mask_s3_uris[j] )
    
    print( file_data['image'][0:10] )
    print( image_s3_uris_temp[0:10] )
    print( len(image_s3_uris) )
    print( len(mask_s3_uris) )
    print( len(image_s3_uris_temp) )
    print( len(mask_s3_uris_temp) )
    if len( file_data['image'] ) != len(image_s3_uris_temp):
        print('Only a portion of the desired files from the json were not found.')

    image_s3_uris = image_s3_uris_temp
    mask_s3_uris = mask_s3_uris_temp

print( [os.path.basename(image_s3_uris[i]) for i in range(10)] )
print( [os.path.basename(mask_s3_uris[i]) for i in range(10)] )

['658372ff0b6bbc0fbbee975a__X17.0228 Y-193.6713 Z117.7757.png', '658372ff0b6bbc0fbbee9776__X17.0229 Y-197.8379 Z117.7757.png', '658372ff0b6bbc0fbbee9792__X17.0229 Y-202.0046 Z117.7757.png', '658373010b6bbc0fbbee9802__X21.1897 Y-193.6713 Z117.7757.png', '658373020b6bbc0fbbee981e__X21.1897 Y-197.8379 Z117.7757.png', '658373020b6bbc0fbbee983a__X21.1898 Y-202.0047 Z117.7757.png', '658373030b6bbc0fbbee9872__X25.3565 Y-193.6713 Z117.7757.png', '658373040b6bbc0fbbee988e__X25.3565 Y-202.0047 Z117.7757.png', '658373050b6bbc0fbbee98e2__X25.3597 Y-197.838 Z117.7759.png', '6583730a0b6bbc0fbbee9a16__X17.0228 Y-193.6713 Z117.9758.png']
['64b1e3703f092c72f7b87e8a-mask.png', '64b1e3703f092c72f7b87ec0-mask.png', '64b1e3723f092c72f7b87ef6-mask.png', '64b1e3733f092c72f7b87f2c-mask.png', '64b1e3733f092c72f7b87f62-mask.png', '64b1e3743f092c72f7b87f98-mask.png', '64b1e3753f092c72f7b88000-mask.png', '64b1e3753f092c72f7b8801b-mask.png', '64b1e3753f092c72f7b88036-mask.png', '64b1e3763f092c72f7b8806c-mask.png']

In [26]:


match_image_mask = False
if match_image_mask: 
    im_basenames = [os.path.basename(image_s3_uris[i])[0:24] for i in range(len(image_s3_uris))]
    mask_basenames = [os.path.basename(mask_s3_uris[i])[0:24] for i in range(len(mask_s3_uris))]

    intersection = list( set(im_basenames) & set(mask_basenames) )

    image_s3_uris_temp = []
    mask_s3_uris_temp = []
    for i, inter in enumerate(intersection):
        for im_path in image_s3_uris:
            if inter in im_path:
                image_s3_uris_temp.append( im_path )

        for ma_path in mask_s3_uris:
            if inter in ma_path: 
                mask_s3_uris_temp.append( ma_path )
    
    im_basenames = sorted( im_basenames )
    mask_basenames = sorted( mask_basenames )
    
    print( [os.path.basename(image_s3_uris_temp[i]) for i in range(10)] )
    print( [os.path.basename(mask_s3_uris_temp[i]) for i in range(10)] )
    
    print( len( image_s3_uris_temp ) )
    print( len( mask_s3_uris_temp ))
    
    image_s3_uris = image_s3_uris_temp
    mask_s3_uris = mask_s3_uris_temp
    
else:
    print( "{} {}".format( len(image_s3_uris), len(mask_s3_uris) ))
    if len(image_s3_uris) != len(mask_s3_uris):
        print('Something is not right!! This is a code work around for not having matching images and masks')
        if len(mask_s3_uris) < len(image_s3_uris):
            for i in range(len(mask_s3_uris), len(image_s3_uris)):
                mask_s3_uris.append( mask_s3_uris[0] )

# in this particular case for RunBatchInference-Copy7 the image and mask s3 buckets do not correspond to one another.
# TODO: Add some filtering to ensure image and mask contain same filenames

1419 1419


### Create JSON and Upload

In [27]:
import json
from pathlib import Path
def upload_json_to_s3(s3_resource, dictionary: dict, s3_uri: str, indent=2) -> None:
    bucket, key = s3_uri.replace("s3://","").split("/", 1)
    obj = s3_resource.Object(bucket, key)
    obj.put(Body=json.dumps(dictionary, indent=indent).encode())


In [28]:
# Location to upload the json s3 files
json_s3_prefix = "s3://baselabelinginfrastack-alllabeledimages/output_inference/paper/{}/json".format(JOB_NAME)

# Ensure json_s3_prefix ends with a "/"
json_s3_prefix = json_s3_prefix if json_s3_prefix.endswith("/") else json_s3_prefix + "/"

s3 = boto3.resource("s3")
print( json_s3_prefix )
#bucket, prefix = json_s3_prefix.replace("s3://").split("/", 1)

manifest_content = [{"prefix": json_s3_prefix}]

for image, mask in zip(image_s3_uris, mask_s3_uris):
    content = {"image": image, "mask": mask}
    image_name = Path(image).stem
    json_uri = f"{json_s3_prefix}{image_name}.json"
    upload_json_to_s3(s3, content, json_uri)
    manifest_content.append(f"{image_name}.json")
    
# Upload the manifest file
manifest_uri = f"{json_s3_prefix}image_mask.manifest"
upload_json_to_s3(s3, manifest_content, manifest_uri)

print(f"Created image_mask manifest file with {len(manifest_content) - 1} pairs:")
print(f"\t uri: {manifest_uri}")

s3://baselabelinginfrastack-alllabeledimages/output_inference/paper/big-training-80-20-20-split---251105-2349--c2-labeled-data/json/
Created image_mask manifest file with 1419 pairs:
	 uri: s3://baselabelinginfrastack-alllabeledimages/output_inference/paper/big-training-80-20-20-split---251105-2349--c2-labeled-data/json/image_mask.manifest


In [29]:
# Name of the model to use for the transform job
# Please see the "PackageModel" model notebook for instructions on how to create a model package.
#model_name = "17k-dataset-cleaned-1--1-it0-241018-1441"
#model_name = "test-inference-job14"
model_name = JOB_NAME#"model_{}".format(JOB_NAME)
#model_name = "17k-dataset-cleaned-geometry-draw-0"

# Instance Deployment Parameters, set the desired instance type and count
instance_type = "ml.c5.2xlarge"  # Ensure instance type has sufficient memory for model
instance_count = 8  # up to 8

# Output
# In s3 uri format e.g.: f"s3://{bucket_name}/{prefix}"
# "s3://bucket-name/prefix/to/output/"
output_path = "s3://baselabelinginfrastack-alllabeledimages/output_inference/paper/{}/output".format(JOB_NAME)

# Whether to wait for job. If true, blocks processing in this notebook and writes log output.
# If False, will return immediately and transform progress can be viewed in 
# SageMaker's Inference/Batch Transforms lists
wait_for_job = False

# Data to be processed. Uncomment and update the block that is appropriate
# For S3 Uri Prefix containing images to be processed
# Same format as output_s3_uri: "s3://bucket-name/path/to/images/"
# 'data' varible is the content printed above in 'print(f"\t uri: {manifest_uri}")'
data = manifest_uri #"s3://baselabelinginfrastack-alllabeledimages/output_inference/job14/image_mask.manifest"
data_type = "ManifestFile"
content_type = "application/json" # streams the raw bytes to the model
max_payload_size = 25 # Maximum payload size in MB. Must be at least as large as a single image


# TODO: Add manifest options

In [30]:
from sagemaker.transformer import Transformer

In [31]:
# The transform object deploys the give model to the desired instance type and count.

batch_transformer = Transformer(
    model_name=model_name,
    instance_type=instance_type, 
    instance_count=instance_count,
    output_path=output_path,
    strategy="SingleRecord",
    max_payload=max_payload_size,
    max_concurrent_transforms=1,
    accept="application/json",
)

In [32]:
# We will point the transform job to an S3 Prefix and it will send all data in the prefix to the job. We can alternatively use a manifest file which lists the specific files to run rather than a prefix.
# See run parameters from 'measurements-partial-1185-missing-run3-240814-1940' 

batch_transformer.transform(
    data=data,
    data_type=data_type,
    content_type=content_type,
    wait=wait_for_job
)

INFO:sagemaker:Creating transform job with name: pytorch-inference-2025-11-06-16-52-36-562


## Update Input Function in Template
Update the inference template's `input_fn` to read in the json files and return bytes for 2 images

In [33]:
JSON_CONTENT_TYPE = "application/json"

def get_s3_object(s3_resource, s3_uri: str):
    bucket, key = s3_uri.replace("s3://", "").split("/", 1)
    try:
        obj = s3_resource.Object(bucket, key).get()
        return obj["Body"].read()
    except s3_resource.meta.client.exceptions.NoSuchKey:
        return None


def input_fn(serialized_input_data, content_type=JSON_CONTENT_TYPE):
    if content_type == JSON_CONTENT_TYPE:
        content = json.loads(serialized_input_data)
        s3_resource = boto3.resource("s3")
        image = get_s3_object(s3_resource, content["image"])
        mask = get_s3_object(s3_resource, content["mask"])
        
        if image is None or mask is None:
            raise ValueError(f"Unable to load either mask or image in set: \n  {json.dumps(content, indent=2)}")
        
        return (image, mask)   
    else:
        raise ValueError(f"Unexpected content type given: {content_type}")